# Self-Pruning Neural Network on CIFAR-10

The idea here is to build a network that figures out which of its own weights are useless — and learns to zero them out *while* training, not after.

Each linear layer gets learnable "gate" scores attached to its weights. During the forward pass, gates (passed through sigmoid) get multiplied onto the weights. If the optimizer decides a gate isn't worth keeping, it pushes it toward zero — effectively removing that connection.

We control how aggressively this happens using a sparsity penalty (λ) in the loss.

**Outline:**
1. `PrunableLinear` — custom linear layer with per-weight gate scores
2. Sparsity loss — L1 penalty on gate values
3. Training on CIFAR-10 with three λ values
4. Accuracy vs sparsity tradeoff analysis

In [19]:
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

print(f"torch version: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")

torch version: 2.4.1+cu121
cuda available: True


## Part 1 — PrunableLinear Layer

This is a custom drop-in for `nn.Linear` that adds a learnable gate to every weight.
The key idea: instead of using the raw weight matrix, we multiply it element-wise by
sigmoid(gate_scores) before doing the linear op. Gates start at roughly 0.6 (not fully open,
not fully closed), and the optimizer can drive them toward 0 to prune connections it doesn't need.

Why sigmoid? It squashes gate values into (0, 1), so they can never go negative or explode.
A gate near 0 means that weight has almost no effect — it's effectively pruned.

In [20]:
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))

        # one gate score per weight, same shape
        # init at 0.0 so sigmoid(0.0) = 0.5 — gates start at the midpoint,
        # where the sigmoid gradient is steepest, so the sparsity loss can
        # move them quickly. Using 0.4 was saturating too early.
        self.gate_scores = nn.Parameter(torch.empty(out_features, in_features))

        self._init_params()

    def _init_params(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.constant_(self.gate_scores, 0.0)

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores)   # values in (0, 1)
        pruned_w = self.weight * gates            # element-wise mask
        return F.linear(x, pruned_w, self.bias)

    def get_gates(self):
        """Detached gate values for inspection."""
        return torch.sigmoid(self.gate_scores).detach()


# quick sanity check
test_layer = PrunableLinear(10, 5)
x_test = torch.randn(3, 10)
out_test = test_layer(x_test)
print(f"output shape: {out_test.shape}")
print(f"gate shape:   {test_layer.get_gates().shape}")
print(f"initial gate mean: {test_layer.get_gates().mean():.4f}")  # should be 0.5

output shape: torch.Size([3, 5])
gate shape:   torch.Size([5, 10])
initial gate mean: 0.5000


## Network Architecture

CIFAR-10 images are 32×32×3, so 3072 input dims when flattened.
Going with a simple 3-layer MLP — nothing fancy, just enough capacity to show pruning effects clearly.

All layers use `PrunableLinear`, so gates can prune connections anywhere in the network.

In [21]:
class PrunableNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 32x32x3 = 3072 input features
        self.fc1 = PrunableLinear(3072, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)      # flatten
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)                # raw logits, CE loss handles softmax
        return x

    def get_all_gates(self):
        """Collect and concatenate gate values from all prunable layers."""
        all_gates = []
        for m in self.modules():
            if isinstance(m, PrunableLinear):
                all_gates.append(m.get_gates().flatten())
        return torch.cat(all_gates)


# count params to make sure things look reasonable
net_test = PrunableNet()
total_params = sum(p.numel() for p in net_test.parameters())
print(f"total parameters: {total_params:,}")
# gate params alone (same count as weights)
gate_params = sum(p.numel() for n, p in net_test.named_parameters() if 'gate' in n)
print(f"gate parameters: {gate_params:,}")

total parameters: 3,413,770
gate parameters: 1,706,496


## Part 2 — Sparsity Loss

The total loss is:

```
Total Loss = CrossEntropy(logits, labels) + λ × SparsityLoss
```

`SparsityLoss` has two components:
- **L1 term:** mean of all gate values — pushes gates toward zero
- **Binarization term:** mean of `gate × (1 - gate)` — penalizes gates stuck in the middle (near 0.5), encouraging them to snap to either 0 or 1

```
SparsityLoss = mean(gates) + mean(gates × (1 - gates))
```

The binarization term is zero when gates are at 0 or 1, and maximal at 0.5.
Together, L1 pushes gates toward zero and the binarization term forces them to commit —
preventing gates from stalling at intermediate values like 0.05 where they still
consume some capacity without being fully pruned.

In [22]:
def sparsity_loss(model):
    """Two-part loss:
    1. L1: mean(gates) — pulls all gates toward zero
    2. Binarization: mean(gates*(1-gates)) — penalizes gates stuck near 0.5,
       forcing them to commit to 0 or 1 rather than stalling in the middle.
    Both terms are normalized so lambda scales the same way regardless of network size."""
    l1_total, binar_total, count = 0.0, 0.0, 0
    for m in model.modules():
        if isinstance(m, PrunableLinear):
            gates = torch.sigmoid(m.gate_scores)
            l1_total    = l1_total    + gates.sum()
            binar_total = binar_total + (gates * (1 - gates)).sum()
            count += gates.numel()
    return (l1_total + binar_total) / count


def total_loss(logits, labels, model, lam):
    ce = F.cross_entropy(logits, labels)
    sp = sparsity_loss(model)
    return ce + lam * sp


# sanity check — at init gates=0.5, binarization term = 0.5*(1-0.5) = 0.25
# so sparsity_loss = (0.5 + 0.25) = 0.75
dummy_model = PrunableNet()
dummy_logits = torch.randn(8, 10)
dummy_labels = torch.randint(0, 10, (8,))
print(f"sparsity term at init: {sparsity_loss(dummy_model).item():.4f}")  # ~0.75

sparsity term at init: 0.7500


## Data Loading — CIFAR-10

Standard CIFAR-10 setup with per-channel normalization.
Using `num_workers=0` here since multiprocessing in notebooks on Windows can be finicky.

In [23]:
def get_cifar10_loaders(batch_size=128):
    # standard CIFAR-10 normalization values (per channel mean/std)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616)
        )
    ])

    train_data = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform
    )
    test_data = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform
    )

    # num_workers=0 to avoid multiprocessing issues in notebooks on Windows
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader  = DataLoader(test_data,  batch_size=256, shuffle=False, num_workers=0)

    return train_loader, test_loader


train_loader, test_loader = get_cifar10_loaders()
print(f"train batches: {len(train_loader)}")
print(f"test batches:  {len(test_loader)}")

Files already downloaded and verified
Files already downloaded and verified
train batches: 391
test batches:  40


In [ ]:
def train_one_epoch(model, loader, optimizer, lam, device):
    model.train()
    running_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = total_loss(logits, labels, model, lam)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


def sparsity_level(model, threshold=0.1):
    """Fraction of weights whose gate value is below threshold.
    A gate below 0.1 contributes <10% of the weight's full value — effectively pruned.
    We use 0.1 rather than 0.01 because sigmoid saturates near zero: gate_scores would
    need to reach -4.6 (sigmoid(-4.6)=0.01) but gradient vanishes well before that.
    Gates that land below 0.1 are genuinely inactive in the forward pass."""
    all_gates = model.get_all_gates()
    return (all_gates < threshold).float().mean().item()


print("training utils defined")

## Part 3 — Training with Three Lambda Values

Training in two phases for each λ:
- **Warmup (5 epochs, λ=0):** network learns to classify first, so weights have useful values before pruning starts
- **Pruning (25 epochs, full λ):** sparsity + binarization pressure applied on an already-functional network

| λ value | Expected effect |
|---------|----------------|
| 2 | Mild pruning — most gates snap to 1, accuracy stays high |
| 8 | Moderate pruning — clear sparsity, some accuracy drop |
| 30 | Aggressive pruning — majority of gates snap to 0 |

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"training on: {device}")

lambdas       = [2, 8, 30]
warmup_epochs = 5
prune_epochs  = 25

results = {}

for lam in lambdas:
    print()
    print(f"=== lambda = {lam} ===")

    model     = PrunableNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=prune_epochs, eta_min=1e-5
    )

    # phase 1: warmup — let weights learn useful features before pruning starts
    for epoch in range(warmup_epochs):
        train_one_epoch(model, train_loader, optimizer, lam=0, device=device)

    # phase 2: apply sparsity pressure
    for epoch in range(prune_epochs):
        t0 = time.time()
        avg_loss = train_one_epoch(model, train_loader, optimizer, lam, device)
        scheduler.step()
        elapsed = time.time() - t0

        if (epoch + 1) % 5 == 0:
            print(f"  epoch {epoch+1:02d}/{prune_epochs}  loss={avg_loss:.4f}  lr={scheduler.get_last_lr()[0]:.2e}  ({elapsed:.1f}s)")

    acc      = evaluate(model, test_loader, device)
    sparsity = sparsity_level(model)
    gates    = model.get_all_gates().cpu()

    results[lam] = {"test_acc": acc, "sparsity": sparsity, "gates": gates}
    print(f"  -> test acc: {acc:.4f}  |  sparsity: {sparsity:.4f}")

print("done with all runs")

training on: cuda

=== lambda = 2 ===
  epoch 05/25  loss=2.0709  lr=9.05e-04  (9.2s)
  epoch 10/25  loss=1.6286  lr=6.58e-04  (9.1s)
  epoch 15/25  loss=1.3652  lr=3.52e-04  (9.2s)
  epoch 20/25  loss=1.2668  lr=1.05e-04  (9.2s)
  epoch 25/25  loss=1.2440  lr=1.00e-05  (9.5s)
  -> test acc: 0.5619  |  sparsity: 0.0000

=== lambda = 8 ===
  epoch 05/25  loss=5.1640  lr=9.05e-04  (9.2s)
  epoch 10/25  loss=4.0638  lr=6.58e-04  (9.8s)
  epoch 15/25  loss=3.4973  lr=3.52e-04  (9.5s)
  epoch 20/25  loss=3.2442  lr=1.05e-04  (10.1s)
  epoch 25/25  loss=3.1723  lr=1.00e-05  (10.1s)
  -> test acc: 0.5663  |  sparsity: 0.0000

=== lambda = 30 ===
  epoch 05/25  loss=11.6200  lr=9.05e-04  (10.2s)
  epoch 10/25  loss=7.6525  lr=6.58e-04  (10.2s)
  epoch 15/25  loss=6.1138  lr=3.52e-04  (10.6s)
  epoch 20/25  loss=5.4906  lr=1.05e-04  (10.8s)
  epoch 25/25  loss=5.3295  lr=1.00e-05  (10.7s)
  -> test acc: 0.5706  |  sparsity: 0.0000
done with all runs


## Results

In [ ]:
print(f"{'lambda':>8}  {'test accuracy':>14}  {'sparsity (gate<0.10)':>21}")
print("-" * 50)
for lam, r in results.items():
    print(f"{lam:>8.0f}  {r['test_acc']:>13.2%}  {r['sparsity']:>20.2%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ["steelblue", "darkorange", "firebrick"]

for ax, (lam, r), color in zip(axes, results.items(), colors):
    gates_np = r["gates"].numpy()
    acc_val, spar_val = r["test_acc"], r["sparsity"]
    ax.hist(gates_np, bins=60, color=color, alpha=0.75, edgecolor="white", linewidth=0.4)
    ax.axvline(x=0.1, color="black", linestyle="--", linewidth=1.2, label="prune threshold (0.10)")
    ax.set_title(f"lambda={lam}  acc={acc_val:.2%}  sparsity={spar_val:.2%}", fontsize=10)
    ax.set_xlabel("gate value (sigmoid output)")
    ax.set_ylabel("number of weights")
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)

fig.suptitle("Gate Value Distribution Across Lambda Settings", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("gate_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("plot saved to gate_distribution.png")

---

See [REPORT.md](REPORT.md) for the full write-up — sparsity explanation, results table, observations, and the gate distribution plot.